In [1]:
import pandas as pd
from Bio import Seq
import numpy as np
import collections
import csv
import itertools
import json
import os

In [2]:
SPACERS = [
    ("all_spacers_cd95_cov100", "all_spacers"),
    ("Cas9_spacers_cd95_cov100", "Cas9_spacers"),
    ("Cas9a_spacers_cd95_cov100", "Cas9a_spacers"),
    ("Cas9a_spy_cluster_cd90_cov80_spacers_cd95_cov100",
     "Cas9a_spy_like_cluster_spacers"),
    ("Cas9a_spy_cluster_spacers_cd90_cov80", "Cas9a_spy_cluster_spacers"),
]
############# ro calculate kmers: Exist in CRISPR_MAAGAD/Burstein_Features/reference? tr in tr_ref?   #################



# PROTOSPACERS = [
#     ("CRISPRIL_spacers_90_id_90_coverage", "targets_90id90cov"),
#     ("CRISPRIL_spacers_95_id_95_coverage", "targets_95id95cov"),
#     ("CRISPRIL_spacers_100_id_100_coverage", "targets_100id100cov")
# ]
# for k in [3,5]:
#     for CT_GA in ['','1']:
#         for spacer_name in ['all_spacers','Cas9_spacers','Cas9a_spacers','Cas9a_spy_like_cluster_spacers','Cas9a_spy_cluster_spacers']:
#             # k=3
#             # CT_GA = '1'
#             # spacers_name='all_spacers'
#             if CT_GA!='':
#                 CT_GA = 'tr_CT_GA-'
#             if k==5:
#                 k_name = 'k_5-'
#                 folder = '/tamir2/shaicohen1/CRISPR_MAAGAD/dudu_burst_feats_concised/5_mers/'
#             elif k==3:
#                 k_name = 'k_3-'
#                 folder = '/tamir2/shaicohen1/CRISPR_MAAGAD/dudu_burst_feats_concised/3_mers/'
#             if spacers_name=='all_spacers':
#                 spacers_full_name = 'all_spacers_cd95_cov100'
#             elif spacers_name=='Cas9_spacers':
#                 spacers_full_name = 'Cas9_spacers_cd95_cov100'
#             elif spacers_name=='Cas9a_spacers':
#                     spacers_full_name = 'Cas9a_spacers_cd95_cov100'
#             elif spacers_name=='Cas9a_spy_like_cluster_spacers':
#                     spacers_full_name = 'Cas9a_spy_cluster_cd90_cov80_spacers_cd95_cov100'
#             elif spacers_name=='Cas9a_spy_cluster_spacers':
#                     spacers_full_name = 'Cas9a_spy_cluster_spacers_cd90_cov80'
#             sorted_tsv_name = folder+k_name+CT_GA+spacers_full_name+'.sorted.tsv'
#             df_kmers= pd.read_csv(sorted_tsv_name,sep='\t')
#             df_kmers['quartile'] = [np.floor((i)/(0.25*df_kmers.shape[0])) for i in df_kmers.index]
#             print(f'{k} {CT_GA} {spacer_name}')
# print('done')

In [32]:
class Quantiler:
    RANKING = {
        "A": 1,
        "C": 2,
        "T": 4,
        "G": 3
    }
    OPPOSITES = {
        "A": "T",
        "C": "G",
        "T": "A",
        "G": "C"
    }

    def __init__(self, k, spacers_name, orig="", new="",CT_GA = ''):
        if CT_GA!='':
            CT_GA = 'tr_CT_GA-'
        if k==5:
            k_name = 'k_5-'
            folder = '/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/Burstein_code/5_mers/'
        elif k==3:
            k_name = 'k_3-'
            folder = '/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/Burstein_code/3_mers/'
        if spacers_name=='all_spacers':
            spacers_full_name = 'all_spacers_cd95_cov100'
            
        elif spacers_name=='Cas9_spacers':
            spacers_full_name = 'Cas9_spacers_cd95_cov100'
            
        elif spacers_name=='Cas9a_spacers':
                spacers_full_name = 'Cas9a_spacers_cd95_cov100'
            
        elif spacers_name=='Cas9a_spy_like_cluster_spacers':
                spacers_full_name = 'Cas9a_spy_cluster_cd90_cov80_spacers_cd95_cov100'
            
        elif spacers_name=='Cas9a_spy_cluster_spacers':
                spacers_full_name = 'Cas9a_spy_cluster_spacers_cd90_cov80'
            
        sorted_json_name = folder+k_name+CT_GA+spacers_full_name+'.quantiles.json'        
        self.from_json(sorted_json_name)
        self.is_tr = bool(CT_GA)

        self.translate = str.maketrans(orig, new)
        self.k = k
        self.number_of_complements = self.get_number_of_complements()
        self.min_energies = None

    def to_json(self):
        with open(self.path.quantiles(self.k), "w") as file:
            json.dump(self.kmers, file)

    def from_json(self,json_path):
        self.kmers = {}
        with open(json_path, "r") as file:
            kmers = json.load(file)
            for kmer, info in kmers.items():
                self.kmers[kmer] = info[0], float(info[1])

    def get_number_of_complements(self):
        bases = [base for base in self.OPPOSITES.keys()]
        possibilities = ["".join(p) for p in itertools.product(bases, repeat=self.k)]
        result = set()
        for possibility in possibilities:
            result.add(self.get_correct_direction_complement(possibility.translate(self.translate), is_tr=self.is_tr))

        # TODO probably ok to remove this - test.
        if self.is_tr:
            for i in result.copy():
                if i[-1::-1] in result and i[-1::-1] != i:
                    result.remove(i)

        return len(result)

    def from_csv(self):
        # df = pd.read_csv('/tamir2/shaicohen1/CRISPR_MAAGAD/dudu_burst_feats_concised/Cas9_spacers_cd95_cov100.sorted.tsv',sep='\t')
        # sorted_csv_name = self.path.sorted_csv(self.k)
        sorted_csv_name = '/tamir2/shaicohen1/CRISPR_MAAGAD/dudu_burst_feats_concised/Cas9_spacers_cd95_cov100.sorted.tsv'
        # cmd("(head -1 {csvfile} && tail -n +2 {csvfile} | sort -k2 -nr) > {sorted}".format(csvfile=self.path.tsv(self.k), sorted=sorted_csv_name))
        with open(sorted_csv_name, "r") as file:
            reader = csv.DictReader(file, delimiter="\t")
            for index, row in enumerate(reader):
                quantile = int((index / self.number_of_complements) * 4)
                self.kmers[row["kmer"]] = quantile, row["log_frequency"]

    @classmethod
    def get_correct_direction_complement(cls, sequence, is_tr=False):
        length = len(sequence)
        if is_tr:
            for i in set(range(int(length / 2) + 1)):
                if cls.RANKING[sequence[i]] < cls.RANKING[sequence[length - i - 1]]:
                    return sequence

                if cls.RANKING[sequence[i]] > cls.RANKING[sequence[length - i - 1]]:
                    return sequence[-1::-1]

            return sequence

        for i in set(range(int(length / 2) + 1)):
            first_rank = cls.RANKING[sequence[i]]
            second_rank = cls.RANKING[cls.OPPOSITES[sequence[length - i - 1]]]
            if first_rank < second_rank:
                return sequence

            if first_rank > second_rank:
                return "".join([cls.OPPOSITES[sequence[length - i - 1]] for i in range(length)])

        return sequence

    def get_number_of_kmers(self, sequence):
        return len(sequence) - self.k + 1

    def count_reverse_complement_kmers(self, sequence):
        mers = collections.defaultdict(int)
        for i in range(self.get_number_of_kmers(sequence)):
            mers[self.get_correct_direction_complement(sequence[i: i + self.k], is_tr=self.is_tr)] += 1

        return mers

    def get_kmers_log_distribution(self, sequence):
        sequence_kmers = self.count_reverse_complement_kmers(sequence.translate(self.translate))
        number_of_mers_in_sequence = self.get_number_of_kmers(sequence)
        occurrences = {j: 0 for j in range(4)}
        frequencies = 0
        for kmer, count in sequence_kmers.items():
            if kmer in self.kmers:
                actual_kmer = kmer
            else:
                actual_kmer = str(Seq.Seq(kmer).reverse_complement())
            if actual_kmer in self.kmers:
                occurrences[self.kmers[kmer][0]] += count
                frequencies += self.kmers[kmer][1] * count
            else:
                print(f'WTF {kmer}')

        return [count / number_of_mers_in_sequence for quantile, count in occurrences.items()] + [round(frequencies,1)]

    def orig_get_kmers_log_distribution(self, sequence):
        sequence_kmers = self.count_reverse_complement_kmers(sequence.translate(self.translate))
        number_of_mers_in_sequence = self.get_number_of_kmers(sequence)
        occurrences = {j: 0 for j in range(4)}
        frequencies = 0
        for kmer, count in sequence_kmers.items():
            if kmer in self.kmers:
                occurrences[self.kmers[kmer][0]] += count
                frequencies += self.kmers[kmer][1] * count

            else:
                occurrences[4 - 1] += count
    
        return [count / number_of_mers_in_sequence for quantile, count in occurrences.items()] + [frequencies]#, sequence_kmers 

In [4]:
# sites_folder = '/tamir2/shaicohen1/CRISPR_MAAGAD/dudu_burst_feats_concised/sites/'
# ics = pd.read_csv(sites_folder + 'ICS_w_isana.csv')
# k562 = pd.read_csv(sites_folder + 'K562_w_isana.csv')
# u937 = pd.read_csv(sites_folder + 'U937_w_isana.csv')
# T = pd.read_csv(sites_folder + 'T_w_isana.csv')
# tomato = pd.read_csv(sites_folder + 'Tomato_w_isana.csv')
# shrimp = pd.read_csv(sites_folder + 'Shrimp_w_isana.csv')
# Leenay = pd.read_csv(sites_folder + 'Leenay_w_isana.csv')
# DeepCRISPR_hek293 = pd.read_csv(sites_folder + 'DeepCRISPR_hek293_w_isana.csv')
# DeepCRISPR_hela = pd.read_csv(sites_folder + 'DeepCRISPR_hela_w_isana.csv')
# DeepHF = pd.read_csv(sites_folder + 'DeepHF_w_isana.csv')

# shrimp = pd.read_csv('/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/moritz_code/shrimp_w_CAI_chimera.csv')
# fly = pd.read_csv('/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/cb_chimera_code/output_sites/chimera_sites/corrected_sites/fly_w_CAI_chimera.csv')
# tomato = pd.read_csv('/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/cb_chimera_code/output_sites/chimera_sites/corrected_sites/tomato_w_CAI_chimera.csv')



In [66]:
raw_t = pd.read_csv('/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/Burstein_code/sites/raw_T_feats.tsv',sep='\t')
raw_t['k5-all_spacers-Q1'].iloc[1]
# raw_t

np.float64(0.5625)

In [63]:
df = pd.read_csv('/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/Burstein_code/sites/T_w_isana.csv')
a = Quantiler(5, 'all_spacers', orig="", new="",CT_GA='')
seqs = df['target_seq'].to_list()
res,kmers = a.orig_get_kmers_log_distribution(seqs[0][:20])
print(res[0])
# print(a.kmers)
q1 = [x for x in a.kmers.keys() if a.kmers[x][0]==0]
len([x for x in kmers if x in q1])/16

0.5625


0.5625

0.1875
defaultdict(<class 'int'>, {'GCACA': 1, 'CACAG': 1, 'ACAGC': 1, 'CAGCA': 1, 'AGCAT': 1, 'AATGC': 1, 'CAATG': 1, 'ATTGG': 1, 'TCCAA': 1, 'GTCCA': 1, 'GGACA': 1, 'GACAC': 1, 'ACACG': 1, 'ACGTG': 1, 'ACGTA': 1, 'CGTAG': 1})


0.1875

In [5]:
# all_dfs = [tomato,shrimp,fly]
# all_dfs_names = ['Tomato','Shrimp','Fly']

tomato = pd.read_csv('/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/cb_chimera_code/output_sites/chimera_sites/corrected_sites/tomato_roots_w_CAI_chimera.csv')
all_dfs = [tomato]
all_dfs_names = ['Tomato_roots']

In [33]:
out_sites_folder = '/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/Burstein_code/output_sites/fly_new_shrimp_tomato/'
# all_dfs = [ics,k562,u937,T,
#            tomato,shrimp,
#            Leenay,DeepCRISPR_hek293,DeepCRISPR_hela,DeepHF]
# all_dfs_names = ['ICS','K562','U937','T',
#                  'Tomato','Shrimp',
#                  'Leenay','DeepCRISPR_hek293','DeepCRISPR_hela','DeepHF']

for i_df in range(len(all_dfs)):
    df = all_dfs[i_df]
    orig_df = df.copy()
    df_name = all_dfs_names[i_df]
    print(df_name)
    cols_to_drop = [x for x in df.columns if 'spacers' in x]
    df=df.drop(columns=cols_to_drop)
    for kkk in  [5,3]:
        for spacer_i in range(5):
            for do_CT_GA in ['','do_CT_GA']:
                spacers = SPACERS[spacer_i][1]
                ugly_spacers = SPACERS[spacer_i][0]
                if do_CT_GA:
                    ugly_GCvAT = '-GCvsAT'
                    ugly_tr_CT_GA = '-tr_CT_GA'
                else:
                    ugly_GCvAT = ''
                    ugly_tr_CT_GA = ''
                a = Quantiler(kkk, spacers, orig="", new="",CT_GA=do_CT_GA)
                cols_to_check = [f'k{kkk}-{spacers}-Q{i}{ugly_GCvAT}' for i in range(1,5)]
                cols_to_check.append(f'k_{kkk}{ugly_tr_CT_GA}-{ugly_spacers}_log_freq_obs{ugly_GCvAT}')
                if do_CT_GA:
                    seqs = df['target_seq'].apply(lambda x: x.replace('C','G').replace('T','A')).to_list()
                else:
                    seqs = df['target_seq'].to_list()
                raw_results = [a.orig_get_kmers_log_distribution(z[:20]) for z in seqs]
                results = [[],[],[],[],[]]
                for i in range(5):
                    results[i] = [x[i] for x in raw_results]
                calc_burstein_df = pd.DataFrame({x:y for x,y in zip(cols_to_check,results)})
                df = pd.concat([df,calc_burstein_df],axis=1)
                # diff = df[cols_to_check]-orig_df[cols_to_check]
                # q=diff<0.00000001#00000001
                # if  ~np.all(q):# and df_name in ['T','','','']:
                #     print(f'diff {df_name} {cols_to_check}')
    df.to_csv(out_sites_folder+f'{df_name}_burst.csv',index=False)
print('done')

Tomato_roots
done


In [20]:
2

2

In [27]:
for x in raw_results:
    print(x[0])
    

[0.1875, 0.0625, 0.0625, 0.6875, 44.59336322158814]
[0.125, 0.25, 0.375, 0.25, 44.6208696075262]
[0.5, 0.25, 0.1875, 0.0625, 41.816270280410976]
[0.3125, 0.375, 0.3125, 0.0, 43.28916579740587]
[0.5625, 0.25, 0.1875, 0.0, 42.12844958776758]
[0.125, 0.25, 0.3125, 0.3125, 44.12098639660448]
[0.5625, 0.3125, 0.125, 0.0, 42.29845377328847]
[0.25, 0.3125, 0.0625, 0.375, 43.944958970353795]
[0.1875, 0.375, 0.4375, 0.0, 43.40591559075156]
[0.375, 0.25, 0.0625, 0.3125, 43.425575634523135]
[0.0, 0.375, 0.3125, 0.3125, 44.35703362723087]
[0.3125, 0.4375, 0.1875, 0.0625, 42.995060039052696]
[0.6875, 0.25, 0.0, 0.0625, 41.46337267425862]
[0.0, 0.3125, 0.375, 0.3125, 44.260233177589924]
[0.125, 0.375, 0.3125, 0.1875, 43.93827748461158]
[0.125, 0.4375, 0.125, 0.3125, 43.92992374510607]
[0.4375, 0.3125, 0.125, 0.125, 42.459751224996]
[0.25, 0.6875, 0.0625, 0.0, 42.7001511235431]
[0.125, 0.4375, 0.1875, 0.25, 43.81054108726514]
[0.4375, 0.25, 0.125, 0.1875, 42.63372787035167]
[0.3125, 0.25, 0.3125, 0.1

In [31]:
raw_results[0][0]

[0.1875, 0.0625, 0.0625, 0.6875, 44.59336322158814]